In [9]:
import os
from dotenv import load_dotenv


In [10]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated,Literal
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage
from pydantic import BaseModel, Field
from langgraph.checkpoint.memory import MemorySaver
import operator

In [11]:
load_dotenv()

True

In [12]:
model = ChatGoogleGenerativeAI(model = "gemini-3.6-flash")

In [13]:
from langgraph.graph.message import add_messages
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage],add_messages]

In [14]:
def chat_node(state:ChatState):
    messages = state['messages']

    response = model.invoke(messages)

    return {'messages':[response]}

In [15]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

chatbot = graph.compile(checkpointer=checkpointer)

In [16]:
initial_state = {'messages':[HumanMessage(content="what is the capital of Nepal?")]}

response=chatbot.invoke(initial_state)

ValueError: Checkpointer requires one or more of the following 'configurable' keys: thread_id, checkpoint_ns, checkpoint_id

In [ ]:
response['messages'][-1].content[0]['text']

'The capital of Nepal is **Kathmandu**.'

In [18]:
thread_id = '1'

while True:
    user_message = input('Type here : ')

    if user_message.strip().lower() in ['exit','quit','bye'] :
        break

    config  = { 'configurable': {'thread_id':thread_id}}
    response = chatbot.invoke({'messages':[HumanMessage(content=user_message)]}, config = config)

    print('AI: ',response['messages'][-1].content[0]['text'])

    

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


AI:  Hello! How can I help you today?
AI:  Nice to meet you, Purush! How can I help you today?
AI:  Your name is Purush!


In [19]:
chatbot.get_state(config=config)

StateSnapshot(values={'messages': [HumanMessage(content='hi', additional_kwargs={}, response_metadata={}, id='6f3ed4da-5235-4e8c-9ddb-1b8908018257'), AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EtUFCtIFAWkUfROoDJsSNoHcRY67ZfCndS5EdEm68Hfp2uCu5uIrnpVWR6whXVZPy8jIRmKLdaJvSLpz/RsGi096lSibwl4wHb5pAaOLdTivSm1x2yneXC7mEYUwzXjkoWGFrqfjVRJigqhfMJPT/ZjRGLjsfr9+cLYqXuXJH3/UHXEk4eThB3vBQJmfRhI5nC//qb44rpFOdq40lW6InlupABMOkj/F3q4uJ4XnVeScD8eN/lzQwTcwJLRdaG3DZrEuli7qy0xE8ttxvs/Hf8yBUEeZQtvJXyXBWoST1a5IjSyi+z+sDLPsFy2ZGT0nyTXiztwMzRU+oMUZYwnBp5mgiNkxKBcro+hu9VvSTodpUwT4dxLN5hJdLkgmrSZxmx1HzBNL2S8IWUlFIQfs9dkMeYWg5kvte1lmOnrdWwGc6qZs9n48FojfthWnIHGG5o9bTst2hO36fLVHYlrjQctmrhX3oob5yub46v/mLIx37ruOTEXKuaHUL4fafCamu5cOx4CaAS5lRL9v0NBN29j5UTMKrP/2L7cZA6xPi425hDyAyN9u8BW4O/4KYjs4RTAOfHKWKbFMNUpb5o3Qwo8UTj0+VeF0VFDk+i+5EmZgxr20k8hF7aW0PuavKp+DGj+Q2A/1jcEWYT8SXnwRsQToSP5HkFyZnWaA7zNYaBXvjqIZsFNQu+chR4FX12qdnWz3uHkCW6mTsp1ODxFpFmeJCgDAUhZCSsbWdozRy